In [ ]:
import warnings
from pathlib import Path
import gc
import sys

import h5py
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import scipy
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from scipy.spatial import Delaunay

from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    adjusted_rand_score,
    adjusted_mutual_info_score,
    completeness_score,
    homogeneity_score,
    normalized_mutual_info_score,
    v_measure_score,
)


warnings.filterwarnings('ignore')
random_seed = 43  # Changed from 42 to 43 (best seed from hyperparameter search)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(random_seed)

from SPROUT import SPROUT_Model, SPROUT_Trainer
from SPROUT.preprocessing import pca, clustering, lsi, extract_coords
from SPROUT.utils import compute_morans_i

In [ ]:
adata1 = "data/E18.5_mouse_brain/E18_adata_rna.h5ad"
adata2 = "data/E18.5_mouse_brain/E18_adata_omics2.h5ad"

adata_omics1 = sc.read_h5ad(adata1)
adata_omics2 = sc.read_h5ad(adata2)

# Spatial coords
coords = extract_coords(adata_omics2)


In [ ]:
# Preprocessing

# RNA
sc.pp.filter_genes(adata_omics1, min_cells=10)
sc.pp.normalize_total(adata_omics1, target_sum=1e4)
sc.pp.log1p(adata_omics1)
sc.pp.highly_variable_genes(adata_omics1, flavor='seurat_v3', n_top_genes=3000)
sc.pp.scale(adata_omics1)
adata_omics1_high = adata_omics1[:, adata_omics1.var['highly_variable']]
rna_features = pca(adata_omics1_high, n_comps=50)


adata_omics2 = adata_omics2[adata_omics1.obs_names].copy() # .obsm['X_lsi'] represents the dimension reduced feature
if 'X_lsi' not in adata_omics2.obsm.keys():
    sc.pp.highly_variable_genes(adata_omics2, flavor="seurat_v3", n_top_genes=3000)
    lsi(adata_omics2, use_highly_variable=False, n_components=51)

adata_omics2.obsm['feat'] = adata_omics2.obsm['X_lsi'].copy()

atac_features = adata_omics2.obsm['feat']

# Extract coordinates
coords = extract_coords(adata_omics2)

# Prepare modality data (keep separate!)
modality_data = [rna_features, atac_features]
input_dims = [rna_features.shape[1], atac_features.shape[1]]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'RNA features: {rna_features.shape}')
print(f'ATAC features: {atac_features.shape}')
print(f'Input dims: {input_dims}')

Device: cpu
RNA features: (2129, 50)
ATAC features: (2129, 50)
Input dims: [50, 50]


In [ ]:
# from sklearn.metrics import (
#     adjusted_rand_score,
#     adjusted_mutual_info_score,
#     completeness_score,
#     homogeneity_score,
#     normalized_mutual_info_score,
#     v_measure_score,
# )

# # Get embeddings and evaluate
# embedding = trainer.get_embedding(coords, modality_data)

# adata = adata_omics2.copy()
# adata.obsm['SPROUT'] = embedding

# n_clusters = 7

# try:
#     clustering(adata, key='SPROUT', add_key='SPROUT_cluster', n_clusters=n_clusters, method='mclust')
#     used_method = 'mclust'
# except Exception as e:
#     print(f'mclust failed: {e}, using kmeans')
#     labels = KMeans(n_clusters=n_clusters, random_state=random_seed, n_init=10).fit_predict(embedding)
#     adata.obs['SPROUT_cluster'] = pd.Categorical(labels.astype(str))
#     used_method = 'kmeans'

# y_true = adata.obs["Combined_Clusters_annotation"].astype(str).values
# y_pred = adata.obs["SPROUT_cluster"].astype(str).values
 
# ari = adjusted_rand_score(y_true, y_pred)
# nmi = normalized_mutual_info_score(y_true, y_pred, average_method="arithmetic")
# ami = adjusted_mutual_info_score(y_true, y_pred)
# homogeneity = homogeneity_score(y_true, y_pred)
# completeness = completeness_score(y_true, y_pred)
# v_measure = v_measure_score(y_true, y_pred)

# print("\n聚类评估指标:")
# print(f"Adjusted Rand Index (ARI): {ari:.4f}")
# print(f"Normalized Mutual Information (NMI): {nmi:.4f}")
# print(f"Adjusted Mutual Information (AMI): {ami:.4f}")
# print(f"Homogeneity : {homogeneity:.4f}")
# print(f"Completeness : {completeness:.4f}")
# print(f"V_measure : {v_measure:.4f}")

fitting ...
  |======================================================================| 100%

聚类评估指标:
Adjusted Rand Index (ARI): 0.2033
Normalized Mutual Information (NMI): 0.3284
Adjusted Mutual Information (AMI): 0.3246
Homogeneity : 0.3305
Completeness : 0.3263
V_measure : 0.3284


In [ ]:
from modal_1.hypergraph import (
    delaunay_star_edges,
    dbscan_edges,
    build_incidence,
    spectral_cluster_from_hypergraph,
)

n = coords.shape[0]

# Spatial hyperedges: Delaunay star (no fixed k)
E_spatial = delaunay_star_edges(coords, s_min=5, s_max=60)

# Expression/embedding hyperedges: DBSCAN with auto eps (no fixed k)
X_embed = np.hstack([rna_features, atac_features])
E_expr, eps_used, expr_lbl = dbscan_edges(
    X_embed,
    min_samples=10,
    s_min=5,
    s_max=600,
    eps=None,  # auto eps by knee
)

# Combine two views into incidence matrix
H, w, cards, edges = build_incidence(n, [E_spatial, E_expr], [0.5, 0.5])

# Hypergraph spectral clustering (Zhou Laplacian, eigengap to pick k)
labels_spec, k_star, eigvals = spectral_cluster_from_hypergraph(
    H,
    w,
    cards,
    r_max=12,
    seed=random_seed,
)

adata = adata_omics2.copy()
adata.obs['HG_cluster'] = pd.Categorical(labels_spec.astype(str))

# Participation statistics: which view dominates each cell
mS = len(E_spatial)
deg_S_raw = np.array(H[:, :mS].sum(axis=1)).ravel()
deg_E_raw = np.array(H[:, mS:].sum(axis=1)).ravel()
pS = (0.5 * deg_S_raw) / (0.5 * deg_S_raw + 0.5 * deg_E_raw + 1e-8)
pE = 1.0 - pS
adata.obs['HG_deg_S'] = deg_S_raw.astype(int)
adata.obs['HG_deg_E'] = deg_E_raw.astype(int)
adata.obs['HG_part_S'] = pS
adata.obs['HG_part_E'] = pE

# Optional evaluation if truth exists
if 'Combined_Clusters_annotation' in adata.obs.columns:
    y_true = adata.obs['Combined_Clusters_annotation'].astype(str).values
    y_pred = adata.obs['HG_cluster'].astype(str).values
    print('ARI', adjusted_rand_score(y_true, y_pred))
    print('NMI', normalized_mutual_info_score(y_true, y_pred, average_method='arithmetic'))
    print('AMI', adjusted_mutual_info_score(y_true, y_pred))
    print('Homogeneity', homogeneity_score(y_true, y_pred))
    print('Completeness', completeness_score(y_true, y_pred))
    print('V_measure', v_measure_score(y_true, y_pred))

print('Spatial edges', len(E_spatial))
print('Expr edges', len(E_expr), 'DBSCAN eps', eps_used)
print('k*', k_star)
